# V25: 8B QLoRA paired attribution pilot

Matched CE and CE+SupCon, 18,000 external passages and two host epochs. Qwen3-8B NF4 with all-linear LoRA; fixed layers12/24/36 readout. Changes from V22 include model size, quantization and adapter coverage together. No submission. LoRA: Hu et al. 2106.09685; SupCon: Khosla et al. 2004.11362; QLoRA: Dettmers et al. 2305.14314 and github.com/artidoro/qlora. Public pool credited to Nishant Dahal and original contributors, CC BY-SA 4.0.

In [ ]:
import subprocess,sys,importlib.metadata
from packaging.version import Version
try: version=importlib.metadata.version('torchao')
except importlib.metadata.PackageNotFoundError: version=None
if version is not None and Version(version)<Version('0.16.0'):
    subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)

import glob,hashlib
from pathlib import Path
wheels=glob.glob('/kaggle/input/**/bitsandbytes-0.50.0-py3-none-manylinux_2_24_x86_64.whl',recursive=True)
assert len(wheels)==1,wheels
assert hashlib.sha256(Path(wheels[0]).read_bytes()).hexdigest()=='173d137610468bec9cddbaa2e049254e97792657ab984e3e737bec1772c1668c'
subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps','--force-reinstall',wheels[0]],check=True)
assert importlib.metadata.version('bitsandbytes')=='0.50.0'


In [ ]:

import os, sys, json, glob, time, subprocess
from pathlib import Path
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[name] = "2"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import importlib.metadata
from packaging.version import Version
try:
    torchao_version = importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    torchao_version = None
if torchao_version is not None and Version(torchao_version) < Version("0.16.0"):
    print({"environment_fix":"remove unused incompatible torchao", "version":torchao_version}, flush=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
import torch, peft, transformers
assert torch.cuda.device_count() == 2, "Paired pilot requires two GPUs"
print({"torch":torch.__version__, "peft":peft.__version__, "transformers":transformers.__version__}, flush=True)
# Check the actual image's adapter integration before loading two full backbones.
tiny = transformers.AutoModel.from_config(transformers.Qwen3Config(
    vocab_size=128, hidden_size=64, intermediate_size=128, num_hidden_layers=2,
    num_attention_heads=4, num_key_value_heads=2, head_dim=16))
tiny = peft.get_peft_model(tiny, peft.LoraConfig(r=4, lora_alpha=8, target_modules=["q_proj","v_proj"]))
tiny(input_ids=torch.tensor([[1,2,3,4]]), use_cache=False).last_hidden_state.sum().backward()
assert any(p.grad is not None and torch.isfinite(p.grad).all() and p.grad.abs().sum()>0
           for n,p in tiny.named_parameters() if "lora_B" in n)
print({"adapter_backward_smoke":"passed"}, flush=True)
del tiny
plan = json.loads("{\"experiment\": \"improvement-v25\", \"kind\": \"8B NF4 all-linear paired CE/SupCon pilot\", \"model\": \"qwen-lm/qwen-3/Transformers/8b-base/1\", \"seed\": 1801, \"pilot_fold\": 0, \"external_cap_per_class\": 1000, \"external_host_cosine_exclusion\": 0.3, \"max_length\": 256, \"batch_size\": 4, \"gradient_accumulation\": 8, \"lora_rank\": 16, \"lora_alpha\": 32, \"target_modules\": \"all-linear\", \"pool_epochs\": 1, \"host_epochs\": 2, \"pool_adapter_lr\": 0.0001, \"pool_head_lr\": 0.0005, \"host_adapter_lr\": 3e-05, \"host_head_lr\": 0.0001, \"contrastive_weight\": 0.1, \"contrastive_temperature\": 0.1, \"variants\": [\"ce\", \"ce_supcon\"], \"checkpoint_rule\": \"Evaluate fixed final host epoch only; no best-epoch selection\", \"promotion\": \"Fixed three-layer readout must reach >=0.4625925926 and fixed 0.15 sparse-pilot blend >=0.5311111111. Select highest blend among qualified arms, report two-arm selection. No direct pilot submission.\", \"wall_limit_seconds\": 25200, \"sources\": [\"https://arxiv.org/html/2004.11362v5\", \"https://github.com/HobbitLong/SupContrast\", \"https://arxiv.org/html/2106.09685v2\", \"https://github.com/microsoft/LoRA\", \"https://huggingface.co/docs/peft/package_reference/lora\", \"https://arxiv.org/abs/2305.14314\", \"https://github.com/artidoro/qlora\", \"https://huggingface.co/docs/peft/developer_guides/quantization\", \"https://huggingface.co/Qwen/Qwen3-8B-Base/raw/main/config.json\"], \"attribution\": \"LoRA and supervised contrastive learning are established methods. Joint CE plus cross-prompt supervised contrastive adaptation here is our experimental combination, not a reproduction of the image-learning paper. Public pool credited to Nishant Dahal and original contributors (CC BY-SA 4.0). QLoRA: Dettmers et al. 2305.14314; implementation uses Hugging Face PEFT/bitsandbytes. Uses ordinary AdamW and fp16 compute on T4.\", \"limitations\": [\"Same repeatedly inspected 540-row fold; not an untouched test\", \"Changes model size, quantization and adapter coverage together; no pure scaling attribution\", \"Two objectives compared; no full-ensemble CV claim\", \"QLoRA instruction-tuning results do not predict attribution accuracy\", \"No paid compute\"], \"comparison\": {\"v22_supcon_readout\": 0.4425925926, \"v22_supcon_sparse_blend\": 0.5111111111}, \"readout\": {\"layers\": [12, 24, 36], \"C\": 0.02, \"max_iter\": 1000, \"seed\": 1901, \"batch_size\": 8}, \"differences_from_v18\": [\"18000 external rows as V22\", \"8B NF4 base with all-linear rank16 adapters instead of 4B fp16 q/v-only\", \"Unchanged fixed readout layers12/24/36 and C0.02\"], \"reason\": \"Test a stronger trainable reader, with matched CE and SupCon objectives; require two-point gains over V22 selected component and sparse blend.\", \"launch_gate\": \"V24 completed finite 8B NF4 updates and adapter reload; projected 5580 training microsteps plus conservative 1800s readout/loading allowance <=24300s (25% below 9h).\", \"quantization\": {\"load_in_4bit\": true, \"quant_type\": \"nf4\", \"double_quant\": true, \"compute_dtype\": \"float16\"}, \"host_sha256\": \"40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707\", \"pool_sha256\": \"37750e899ff6634c147daa7888bfed048e92d3fb1381fe5d21f93bf72a84e8ec\", \"folds_sha256\": \"c64582951ffc6bd6a8233f97775fede7404a52ee5b73c88fcf18b8271fc80508\"}")
fold_by_prompt = json.loads("{\"p000\": 1, \"p001\": 4, \"p002\": 3, \"p003\": 3, \"p004\": 1, \"p005\": 2, \"p006\": 3, \"p007\": 2, \"p008\": 3, \"p009\": 0, \"p010\": 1, \"p011\": 1, \"p012\": 0, \"p013\": 3, \"p014\": 4, \"p015\": 1, \"p016\": 1, \"p017\": 3, \"p018\": 0, \"p019\": 0, \"p020\": 4, \"p021\": 4, \"p022\": 1, \"p023\": 2, \"p024\": 2, \"p025\": 2, \"p026\": 0, \"p027\": 1, \"p028\": 1, \"p029\": 0, \"p030\": 0, \"p031\": 0, \"p032\": 1, \"p033\": 2, \"p034\": 2, \"p035\": 2, \"p036\": 0, \"p037\": 4, \"p038\": 3, \"p039\": 2, \"p040\": 1, \"p041\": 4, \"p042\": 1, \"p043\": 2, \"p044\": 1, \"p045\": 0, \"p046\": 3, \"p047\": 2, \"p048\": 4, \"p049\": 2, \"p050\": 3, \"p051\": 1, \"p052\": 4, \"p053\": 2, \"p054\": 3, \"p055\": 0, \"p056\": 0, \"p057\": 4, \"p058\": 4, \"p059\": 3, \"p060\": 1, \"p061\": 3, \"p062\": 2, \"p063\": 3, \"p064\": 0, \"p065\": 1, \"p066\": 1, \"p067\": 1, \"p068\": 0, \"p069\": 0, \"p070\": 2, \"p071\": 4, \"p072\": 3, \"p073\": 0, \"p074\": 4, \"p075\": 1, \"p076\": 0, \"p077\": 2, \"p078\": 0, \"p079\": 3, \"p080\": 2, \"p081\": 1, \"p082\": 0, \"p083\": 3, \"p084\": 2, \"p085\": 1, \"p086\": 1, \"p087\": 4, \"p088\": 4, \"p089\": 3, \"p090\": 4, \"p091\": 4, \"p092\": 4, \"p093\": 2, \"p094\": 2, \"p095\": 2, \"p096\": 1, \"p097\": 2, \"p098\": 3, \"p099\": 4, \"p100\": 3, \"p101\": 2, \"p102\": 4, \"p103\": 4, \"p104\": 0, \"p105\": 1, \"p106\": 4, \"p107\": 4, \"p108\": 0, \"p109\": 1, \"p110\": 0, \"p111\": 2, \"p112\": 3, \"p113\": 2, \"p114\": 2, \"p115\": 3, \"p116\": 4, \"p117\": 2, \"p118\": 0, \"p119\": 3, \"p120\": 3, \"p121\": 4, \"p122\": 1, \"p123\": 2, \"p124\": 4, \"p125\": 3, \"p126\": 3, \"p127\": 0, \"p128\": 0, \"p129\": 4, \"p130\": 4, \"p131\": 0, \"p132\": 0, \"p133\": 1, \"p134\": 3, \"p135\": 3, \"p136\": 3, \"p137\": 1, \"p138\": 2, \"p139\": 3, \"p140\": 4, \"p141\": 0, \"p142\": 1, \"p143\": 0, \"p144\": 4, \"p145\": 0, \"p146\": 1, \"p147\": 3, \"p148\": 2, \"p149\": 4}")
Path("worker.py").write_text('"""Standalone GPU worker for the frozen v18 paired-objective pilot."""\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\ndef supcon_loss(features, labels, temperature=0.1):\n    """Mean log-probability of same-class positives, excluding self pairs."""\n    z = F.normalize(features.float(), dim=1)\n    logits = z @ z.T / temperature\n    diagonal = torch.eye(len(z), dtype=torch.bool, device=z.device)\n    positives = labels[:, None].eq(labels[None, :]) & ~diagonal\n    counts = positives.sum(1)\n    if not bool((counts > 0).all()):\n        raise ValueError("Every contrastive anchor must have a distinct positive.")\n    denominator = torch.logsumexp(logits.masked_fill(diagonal, -torch.inf), dim=1)\n    return -((logits - denominator[:, None]).masked_fill(~positives, 0).sum(1) / counts).mean()\n\n\ndef paired_batches(frame, steps, seed):\n    """Two authors, two different prompts each; identical schedule for both arms."""\n    rng = np.random.default_rng(seed)\n    groups = {label: {p: sub.index.to_numpy() for p, sub in group.groupby("group")}\n              for label, group in frame.groupby("label")}\n    assert all(len(g) >= 2 for g in groups.values())\n    labels = sorted(groups)\n    for _ in range(steps):\n        selected = []\n        for label in rng.choice(labels, 2, replace=False):\n            for prompt in rng.choice(list(groups[label]), 2, replace=False):\n                selected.append(int(rng.choice(groups[label][prompt])))\n        yield selected\n\n\nclass AttributionModel(nn.Module):\n    def __init__(self, model_dir, plan):\n        super().__init__()\n        from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\n        from transformers import AutoModel, BitsAndBytesConfig\n        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\n            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)\n        backbone = AutoModel.from_pretrained(model_dir, torch_dtype=torch.float16,\n            quantization_config=quant, device_map={"":0},\n            local_files_only=True, attn_implementation="sdpa")\n        assert backbone.config.hidden_size == 4096 and backbone.config.num_hidden_layers == 36\n        backbone = prepare_model_for_kbit_training(backbone,\n            gradient_checkpointing_kwargs={"use_reentrant":False})\n        backbone.config.use_cache = False\n        backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        self.encoder = get_peft_model(backbone, LoraConfig(r=plan["lora_rank"],\n            lora_alpha=plan["lora_alpha"], target_modules=plan["target_modules"], lora_dropout=0.0,\n            bias="none"))\n        width = backbone.config.hidden_size\n        self.norm = nn.LayerNorm(width)\n        self.head = nn.Linear(width, 18)\n        self.projection = nn.Sequential(nn.Linear(width, 256), nn.GELU(), nn.Linear(256, 128))\n\n    def forward(self, batch):\n        hidden = self.encoder(**batch, use_cache=False).last_hidden_state.float()\n        mask = batch["attention_mask"].clone()\n        mask[:, 0] = 0\n        pooled = (hidden * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1)\n        features = self.norm(pooled)\n        return self.head(features), self.projection(features)\n\n\ndef main():\n    import pandas as pd\n    from transformers import AutoTokenizer\n    import transformers, peft\n    plan = json.loads(Path("plan.json").read_text())\n    variant = os.environ["V18_VARIANT"]\n    assert variant in plan["variants"]\n    out = Path(variant)\n    out.mkdir(exist_ok=True)\n    started = time.monotonic()\n    def log(**kw):\n        record = dict(variant=variant, seconds=round(time.monotonic()-started, 2), **kw)\n        print(json.dumps(record), flush=True)\n        with (out / "progress.jsonl").open("a") as f:\n            f.write(json.dumps(record)+"\\n")\n    assert torch.cuda.device_count() == 1\n    torch.manual_seed(plan["seed"])\n    np.random.seed(plan["seed"])\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\n    model_dir = plan["model_dir"]\n    tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)\n    tok.padding_side = "right"\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    frame = pd.read_parquet("training_frame.parquet")\n    valid = pd.read_parquet("validation_frame.parquet")\n    assert set(frame.loc[frame.stage == "host", "group"]).isdisjoint(set(valid.group))\n    def tokenize(df):\n        enc = tok(df.text.tolist(), truncation=True, max_length=plan["max_length"]-1,\n                  add_special_tokens=False)\n        return [ids+[tok.eos_token_id] for ids in enc["input_ids"]]\n    tokens, val_tokens = tokenize(frame), tokenize(valid)\n    def batch(ids, bank):\n        return {k: v.cuda() for k,v in tok.pad({"input_ids": [bank[i] for i in ids]},\n                                              padding=True, return_tensors="pt").items()}\n    model = AttributionModel(model_dir, plan)\n    model.norm.cuda(); model.head.cuda(); model.projection.cuda()\n    trainable = [p for p in model.parameters() if p.requires_grad]\n    # Frozen base is fp16; trainable adapters and heads must be fp32 for GradScaler.\n    for p in trainable:\n        p.data = p.data.float()\n    log(event="loaded", trainable_parameters=sum(p.numel() for p in trainable),\n        torch=torch.__version__, transformers=transformers.__version__, peft=peft.__version__,\n        gpu=torch.cuda.get_device_name(0), training_rows=len(frame), validation_rows=len(valid))\n    scaler = torch.amp.GradScaler("cuda", init_scale=1024)\n    skipped = 0\n    for stage, epochs in [("pool", plan["pool_epochs"]), ("host", plan["host_epochs"])]:\n        subset = frame.loc[frame.stage == stage]\n        micro_steps = math.ceil(len(subset)/plan["batch_size"])\n        optimizer = torch.optim.AdamW([\n            {"params": [p for p in model.encoder.parameters() if p.requires_grad], "lr": plan[stage+"_adapter_lr"]},\n            {"params": list(model.norm.parameters())+list(model.head.parameters())+list(model.projection.parameters()),\n             "lr": plan[stage+"_head_lr"]}], weight_decay=0.01)\n        for epoch in range(epochs):\n            model.train()\n            optimizer.zero_grad(set_to_none=True)\n            losses = []\n            for step, ids in enumerate(paired_batches(subset, micro_steps, plan["seed"] + epoch + (100 if stage == "host" else 0))):\n                if time.monotonic()-started > plan["wall_limit_seconds"]:\n                    raise TimeoutError("Frozen pilot wall budget exhausted; no complete result claimed")\n                labels = torch.as_tensor(frame.loc[ids, "label"].to_numpy(), device="cuda")\n                with torch.autocast("cuda", dtype=torch.float16):\n                    logits, features = model(batch(ids, tokens))\n                    ce = F.cross_entropy(logits.float(), labels)\n                    contrastive = supcon_loss(features, labels, plan["contrastive_temperature"])\n                    loss = ce + (plan["contrastive_weight"] * contrastive if variant == "ce_supcon" else 0)\n                if not torch.isfinite(loss):\n                    raise FloatingPointError("Nonfinite training loss")\n                window_start = (step // plan["gradient_accumulation"]) * plan["gradient_accumulation"]\n                divisor = min(plan["gradient_accumulation"], micro_steps-window_start)\n                scaler.scale(loss/divisor).backward()\n                losses.append(float(loss.detach()))\n                if (step+1) % plan["gradient_accumulation"] == 0 or step+1 == micro_steps:\n                    scaler.unscale_(optimizer)\n                    torch.nn.utils.clip_grad_norm_(trainable, 1.0)\n                    before = scaler.get_scale()\n                    scaler.step(optimizer)\n                    scaler.update()\n                    skipped += int(scaler.get_scale() < before)\n                    optimizer.zero_grad(set_to_none=True)\n                if step % 50 == 0:\n                    log(event="train", stage=stage, epoch=epoch, step=step, steps=micro_steps,\n                        mean_loss=float(np.mean(losses[-50:])), ce=float(ce.detach()),\n                        contrastive=float(contrastive.detach()), skipped_steps=skipped,\n                        peak_gb=torch.cuda.max_memory_allocated()/2**30)\n            log(event="epoch_complete", stage=stage, epoch=epoch, mean_loss=float(np.mean(losses)))\n        checkpoint = out / ("external_only" if stage == "pool" else "final")\n        model.encoder.save_pretrained(checkpoint / "adapter")\n        torch.save({"norm": model.norm.state_dict(), "head": model.head.state_dict(),\n                    "projection": model.projection.state_dict()}, checkpoint / "heads.pt")\n    model.eval()\n    scores = []\n    with torch.inference_mode():\n        for start in range(0, len(valid), 4):\n            ids = list(range(start, min(start+4, len(valid))))\n            with torch.autocast("cuda", dtype=torch.float16):\n                logits, _ = model(batch(ids, val_tokens))\n            scores.append(logits.float().cpu().numpy())\n    scores = np.concatenate(scores)\n    assert np.isfinite(scores).all()\n    np.savez_compressed(out / "validation.npz", ids=valid.id.to_numpy(dtype=str),\n                        labels=np.asarray(plan["labels"], dtype=str), logits=scores,\n                        targets=valid.label.to_numpy())\n    result = {"variant":variant, "accuracy":float((scores.argmax(1)==valid.label.to_numpy()).mean()),\n              "seconds":time.monotonic()-started, "skipped_steps":skipped,\n              "validation_rows":len(valid), "completed":True}\n    (out / "result.json").write_text(json.dumps(result, indent=2))\n    log(event="complete", result=result)\n\n\nif __name__ == "__main__":\n    main()\n')
Path("prepare_data.py").write_text('"""Deterministic training-frame preparation and conservative external overlap screen."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport unicodedata\nimport numpy as np\nimport pandas as pd\nfrom sklearn.feature_extraction.text import TfidfVectorizer\n\n\ndef prepare(host_path, pool_path, fold_by_prompt, plan, output):\n    output = Path(output)\n    output.mkdir(exist_ok=True, parents=True)\n    host = pd.read_csv(host_path, dtype={"id":str, "prompt_id":str})\n    pool = pd.read_parquet(pool_path)\n    assert hashlib.sha256(Path(host_path).read_bytes()).hexdigest() == plan["host_sha256"]\n    assert hashlib.sha256(Path(pool_path).read_bytes()).hexdigest() == plan["pool_sha256"]\n    assert set(host.prompt_id) == set(fold_by_prompt)\n    labels = sorted(host.model.unique())\n    assert len(labels) == 18 and set(pool.model) <= set(labels)\n    def normalize(s):\n        return re.sub(r"\\s+", " ", unicodedata.normalize("NFKC", str(s))).strip().lower()\n    hashes = pool.text.map(normalize)\n    ambiguous = pool.assign(key=hashes).groupby("key").model.nunique()\n    reject = hashes.isin(set(host.text.map(normalize))) | hashes.isin(ambiguous[ambiguous>1].index) | hashes.duplicated()\n    print(json.dumps({"event":"screen_start", "pool_rows":len(pool)}), flush=True)\n    vectorizer = TfidfVectorizer(stop_words="english", sublinear_tf=True, dtype=np.float32)\n    X = vectorizer.fit_transform(pd.concat([host.text, pool.text], ignore_index=True))\n    H, P = X[:len(host)], X[len(host):]\n    similarity = np.zeros(len(pool), dtype=np.float32)\n    for start in range(0, len(pool), 512):\n        similarity[start:start+512] = (P[start:start+512] @ H.T).max(axis=1).toarray().ravel()\n    reject |= similarity >= plan["external_host_cosine_exclusion"]\n    clean = pool.loc[~reject]\n    selected = pd.concat([g.sample(min(len(g), plan["external_cap_per_class"]), random_state=plan["seed"])\n                          for _,g in clean.groupby("model")], ignore_index=True)\n    selected["group"] = "external:" + selected.source.astype(str) + ":" + selected.prompt_key.astype(str)\n    selected["stage"] = "pool"\n    host["group"] = "host:" + host.prompt_id\n    host["stage"] = "host"\n    valid_mask = host.prompt_id.map(fold_by_prompt) == plan["pilot_fold"]\n    train = pd.concat([selected, host.loc[~valid_mask]], ignore_index=True)\n    valid = host.loc[valid_mask].reset_index(drop=True)\n    for frame in [train, valid]:\n        frame["label"] = frame.model.map({s:i for i,s in enumerate(labels)}).astype(int)\n    train.to_parquet(output / "training_frame.parquet", index=False)\n    valid.to_parquet(output / "validation_frame.parquet", index=False)\n    audit = {"pool_rows":len(pool), "excluded_rows":int(reject.sum()),\n             "near_host_rows":int((similarity >= plan["external_host_cosine_exclusion"]).sum()),\n             "selected_pool_rows":len(selected), "host_fit_rows":int((~valid_mask).sum()),\n             "host_validation_rows":int(valid_mask.sum()), "labels":labels,\n             "training_frame_sha256":hashlib.sha256((output / "training_frame.parquet").read_bytes()).hexdigest(),\n             "validation_frame_sha256":hashlib.sha256((output / "validation_frame.parquet").read_bytes()).hexdigest()}\n    (output / "data-audit.json").write_text(json.dumps(audit, indent=2))\n    print(json.dumps(audit), flush=True)\n    return labels\n')
from prepare_data import prepare
host_paths = glob.glob("/kaggle/input/**/train.csv", recursive=True)
host = next(p for p in host_paths if "name-that-llm" in p)
pool_paths = glob.glob("/kaggle/input/**/pool_v4.parquet", recursive=True)
assert len(pool_paths) == 1
model_paths = [p for p in glob.glob("/kaggle/input/**/config.json", recursive=True) if "/8b-base/" in p.lower()]
assert len(model_paths) == 1
plan["model_dir"] = str(Path(model_paths[0]).parent)
plan["labels"] = prepare(host, pool_paths[0], fold_by_prompt, plan, ".")
Path("plan.json").write_text(json.dumps(plan, indent=2))
processes = []
for gpu, variant in enumerate(plan["variants"]):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), V18_VARIANT=variant)
    # Inherit stdout/stderr so Kaggle records actual worker progress and exceptions.
    processes.append((variant, subprocess.Popen([sys.executable, "-u", "worker.py"], env=env)))
try:
    while any(p.poll() is None for _,p in processes):
        failed = [(v,p.returncode) for v,p in processes if p.poll() not in (None,0)]
        if failed:
            raise RuntimeError(f"Paired worker failed: {failed}")
        time.sleep(10)
    failed = [(v,p.returncode) for v,p in processes if p.returncode != 0]
    if failed:
        raise RuntimeError(f"Paired worker failed: {failed}")
except BaseException:
    for _,p in processes:
        if p.poll() is None: p.terminate()
    for _,p in processes: p.wait()
    raise
results = {v:json.loads(Path(v,"result.json").read_text()) for v,_ in processes}
gain = results["ce_supcon"]["accuracy"] - results["ce"]["accuracy"]
summary = {"results":results, "gain":gain, "kind":"learned-head diagnostic only", "submission_authorized":False}
Path("v25_head_diagnostic.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary), flush=True)


In [ ]:

import hashlib
training_plan=json.loads(Path('plan.json').read_text())
Path('training-plan.json').write_text(json.dumps(training_plan,indent=2))
readout_plan=dict(training_plan)
readout_plan.update({'layers': [12, 24, 36], 'C': 0.02, 'max_iter': 1000, 'seed': 1901, 'batch_size': 8})
readout_plan['upstream_path']='.'
readout_plan['host_path']=host
readout_plan['adapter_sha256']={v:hashlib.sha256(Path(p).read_bytes()).hexdigest() for v,p in {
    'ce_final':'ce/final/adapter/adapter_model.safetensors',
    'supcon_final':'ce_supcon/final/adapter/adapter_model.safetensors'}.items()}
Path('readout-plan.json').write_text(json.dumps(readout_plan,indent=2))
Path('plan.json').write_text(json.dumps(readout_plan,indent=2))
Path('readout_worker.py').write_text('"""Matched frozen readouts for base and v18 adapters; one fixed host fold."""\nimport gc\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport time\nimport warnings\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig\nfrom peft import PeftModel, prepare_model_for_kbit_training\n\n\ndef main():\n    plan = json.loads(Path("plan.json").read_text())\n    host = pd.read_csv(plan["host_path"], dtype={"id":str})\n    upstream = Path(plan["upstream_path"])\n    original_plan = json.loads((upstream / "plan.json").read_text())\n    assert original_plan["pilot_fold"] == plan["pilot_fold"]\n    fitting = pd.read_parquet(upstream / "training_frame.parquet")\n    validation = pd.read_parquet(upstream / "validation_frame.parquet")\n    fit_ids = set(fitting.loc[fitting.stage == "host", "id"])\n    val_ids = set(validation.id)\n    assert fit_ids.isdisjoint(val_ids) and fit_ids | val_ids == set(host.id)\n    assert len(fit_ids) == 2159 and len(val_ids) == 540\n    train_mask = host.id.isin(fit_ids).to_numpy()\n    valid_mask = host.id.isin(val_ids).to_numpy()\n    assert host.loc[valid_mask,"id"].tolist() == validation.id.tolist()\n    labels = np.asarray(original_plan["labels"])\n    assert np.array_equal(labels, np.sort(host.model.unique()))\n    tok = AutoTokenizer.from_pretrained(plan["model_dir"], local_files_only=True)\n    tok.padding_side = "right"\n    if tok.pad_token_id is None: tok.pad_token = tok.eos_token\n    tokenized = tok(host.text.tolist(), truncation=True, max_length=plan["max_length"]-1,\n                    add_special_tokens=False)["input_ids"]\n    tokenized = [ids+[tok.eos_token_id] for ids in tokenized]\n    order = np.argsort([len(ids) for ids in tokenized], kind="stable")\n    checkpoint_paths = {"ce_external":"ce/external_only/adapter", "ce_final":"ce/final/adapter",\n                        "supcon_external":"ce_supcon/external_only/adapter", "supcon_final":"ce_supcon/final/adapter"}\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\n    assert torch.cuda.device_count() == 1\n    for variant in json.loads(os.environ["V19_VARIANTS"]):\n        started = time.monotonic()\n        print(json.dumps({"variant":variant,"event":"loading"}), flush=True)\n        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\n            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)\n        model = AutoModel.from_pretrained(plan["model_dir"], torch_dtype=torch.float16,\n            quantization_config=quant, device_map={"":0},\n            attn_implementation="sdpa", local_files_only=True)\n        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)\n        if variant != "base":\n            checkpoint = upstream / checkpoint_paths[variant]\n            weight = checkpoint / "adapter_model.safetensors"\n            assert hashlib.sha256(weight.read_bytes()).hexdigest() == plan["adapter_sha256"][variant]\n            model = PeftModel.from_pretrained(model, checkpoint, is_trainable=False)\n        model = model.eval()\n        width = model.config.hidden_size\n        features = np.empty((len(host),len(plan["layers"]),width), dtype=np.float32)\n        with torch.inference_mode():\n            for start in range(0,len(host),plan["batch_size"]):\n                ids = order[start:start+plan["batch_size"]]\n                enc = {k:v.cuda() for k,v in tok.pad({"input_ids":[tokenized[i] for i in ids]},\n                                                    padding=True,return_tensors="pt").items()}\n                with torch.autocast("cuda", dtype=torch.float16):\n                    states = model(**enc, output_hidden_states=True, use_cache=False).hidden_states\n                mask = enc["attention_mask"].clone(); mask[:,0] = 0\n                for j,layer in enumerate(plan["layers"]):\n                    h = states[layer].float()\n                    features[ids,j] = ((h*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)).cpu().numpy()\n                del states\n                if start % 400 == 0:\n                    print(json.dumps({"variant":variant,"completed":min(start+len(ids),len(host)),\n                                      "total":len(host),"seconds":time.monotonic()-started}), flush=True)\n        assert np.isfinite(features).all()\n        np.save(variant+"_features.npy", features)\n        del model\n        gc.collect(); torch.cuda.empty_cache()\n        predictions = []\n        diagnostics = {}\n        for j,layer in enumerate(plan["layers"]):\n            scaler = StandardScaler()\n            X = scaler.fit_transform(features[train_mask,j])\n            V = scaler.transform(features[valid_mask,j])\n            clf = LogisticRegression(C=plan["C"], max_iter=plan["max_iter"], random_state=plan["seed"])\n            with warnings.catch_warnings():\n                warnings.simplefilter("error", ConvergenceWarning)\n                clf.fit(X,host.loc[train_mask,"model"])\n            assert np.array_equal(clf.classes_, labels)\n            scores = clf.predict_log_proba(V)\n            predictions.append(scores)\n            diagnostics[str(layer)] = float((labels[scores.argmax(1)]==host.loc[valid_mask,"model"]).mean())\n        scores = np.mean(predictions,axis=0)\n        accuracy = float((labels[scores.argmax(1)]==host.loc[valid_mask,"model"]).mean())\n        np.savez_compressed(variant+"_validation.npz", ids=host.loc[valid_mask,"id"].to_numpy(dtype=str),\n                            labels=labels.astype(str), scores=scores, per_layer_scores=np.stack(predictions))\n        result = {"variant":variant,"accuracy":accuracy,"per_layer_accuracy":diagnostics,\n                  "seconds":time.monotonic()-started,"fit_rows":int(train_mask.sum()),\n                  "validation_rows":int(valid_mask.sum()),"feature_shape":list(features.shape),\n                  "feature_sha256":hashlib.sha256(Path(variant+"_features.npy").read_bytes()).hexdigest()}\n        Path(variant+"_result.json").write_text(json.dumps(result,indent=2))\n        print(json.dumps(result),flush=True)\n        del features\n        gc.collect()\n\n\nif __name__ == "__main__":\n    main()\n')
processes=[]
for gpu,variant in enumerate(['ce_final','supcon_final']):
    env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu),V19_VARIANTS=json.dumps([variant]))
    processes.append(subprocess.Popen([sys.executable,'-u','readout_worker.py'],env=env))
try:
    while any(p.poll() is None for p in processes):
        if any(p.poll() not in (None,0) for p in processes):raise RuntimeError('Scaled readout failed')
        time.sleep(10)
    assert all(p.returncode==0 for p in processes)
except BaseException:
    for p in processes:
        if p.poll() is None:p.terminate()
    for p in processes:p.wait()
    raise
finally:
    Path('plan.json').write_text(json.dumps(training_plan,indent=2))
summary={'experiment':'improvement-v25','readouts':{v:json.loads(Path(v+'_result.json').read_text())
         for v in ['ce_final','supcon_final']},'decision':'Requires local fixed-weight sparse blend check',
         'submission_authorized':False}
Path('v25_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary),flush=True)
